# Домашнее задание. Урок 23. Библиотека Pandas

## Задача

В домашнем задании к уроку 12 был написан SQL-запрос, решающий практическую задачу:
для каждой страны определить, кто из клиентов формирует выручку на этом рынке.
Требуется повторить тот же расчёт средствами Pandas.

## Воспроизводимый SQL-запрос

```sql
WITH customer_revenue AS (
        SELECT
            co.country,
            ci.city,
            c.customer_id,
            c.first_name || ' ' || c.last_name AS customer_name,
            COUNT(p.payment_id) AS payments_count,
            SUM(p.amount)       AS total_amount
        FROM customer AS c
        INNER JOIN address AS a  ON a.address_id  = c.address_id
        INNER JOIN city    AS ci ON ci.city_id    = a.city_id
        INNER JOIN country AS co ON co.country_id = ci.country_id
        INNER JOIN payment AS p  ON p.customer_id = c.customer_id
        WHERE p.amount > 0
        GROUP BY co.country, ci.city, c.customer_id, c.first_name, c.last_name
    )
SELECT
    country, city, customer_name, payments_count, total_amount,
    RANK() OVER (PARTITION BY country ORDER BY total_amount DESC) AS rank_in_country,
    CASE
        WHEN total_amount >= 150 THEN 'ключевой'
        WHEN total_amount >= 100 THEN 'постоянный'
        ELSE 'обычный'
    END AS segment
FROM customer_revenue
ORDER BY country, rank_in_country;
```

## Соответствие конструкций

| SQL | Pandas |
|---|---|
| `INNER JOIN ... ON` | `merge(..., on=..., how='inner')` |
| `WHERE` | `query()` |
| `GROUP BY` + `COUNT`/`SUM` | `groupby().agg()` |
| `RANK() OVER (PARTITION BY ...)` | `groupby()[...].rank()` |
| `CASE WHEN` | `np.where()` |
| `ORDER BY` | `sort_values()` |

Нужные таблицы выгружены из базы dvdrental в формате CSV и лежат в подпапке `data`.

In [1]:
import pandas as pd
import numpy as np

print("pandas:", pd.__version__)
print("numpy :", np.__version__)

pandas: 3.0.3
numpy : 2.4.6


## Шаг 1. Загрузка выгруженных таблиц

In [2]:
customer = pd.read_csv("data/customer.csv")
address  = pd.read_csv("data/address.csv")
city     = pd.read_csv("data/city.csv")
country  = pd.read_csv("data/country.csv")
payment  = pd.read_csv("data/payment.csv")

for name, df in [("customer", customer), ("address", address), ("city", city),
                 ("country", country), ("payment", payment)]:
    print(f"{name:9} строк: {df.shape[0]:6}  колонок: {df.shape[1]}")

customer  строк:    599  колонок: 10
address   строк:    603  колонок: 8
city      строк:    600  колонок: 4
country   строк:    109  колонок: 3
payment   строк:  14596  колонок: 6


In [3]:
# Посмотрим, какие колонки нам понадобятся для соединения
print("customer:", list(customer.columns))
print("address :", list(address.columns))
print("payment :", list(payment.columns))

customer: ['customer_id', 'store_id', 'first_name', 'last_name', 'email', 'address_id', 'activebool', 'create_date', 'last_update', 'active']
address : ['address_id', 'address', 'address2', 'district', 'city_id', 'postal_code', 'phone', 'last_update']
payment : ['payment_id', 'customer_id', 'staff_id', 'rental_id', 'amount', 'payment_date']


## Шаг 2. WHERE — отбираем платежи с ненулевой суммой

В SQL это условие `WHERE p.amount > 0`. В Pandas тот же отбор делает метод `query`,
который принимает условие строкой и читается почти как SQL.

In [4]:
payment_positive = payment.query("amount > 0")

print("Платежей всего:        ", payment.shape[0])
print("С ненулевой суммой:    ", payment_positive.shape[0])
print("Отброшено нулевых:     ", payment.shape[0] - payment_positive.shape[0])

Платежей всего:         14596
С ненулевой суммой:     14572
Отброшено нулевых:      24


## Шаг 3. JOIN — собираем цепочку от клиента до страны

Страна не хранится у клиента напрямую: у клиента есть адрес, у адреса город,
у города страна. Поэтому соединений четыре.

Метод `merge` по умолчанию выполняет внутреннее соединение, но параметр `how="inner"`
указан явно — так видно, что это именно аналог `INNER JOIN`.

In [5]:
# берём из каждой таблицы только нужные колонки, чтобы результат не разрастался
data = (
    customer[["customer_id", "first_name", "last_name", "address_id"]]
    .merge(address[["address_id", "city_id"]], on="address_id", how="inner")
    .merge(city[["city_id", "city", "country_id"]], on="city_id", how="inner")
    .merge(country[["country_id", "country"]], on="country_id", how="inner")
    .merge(payment_positive[["customer_id", "payment_id", "amount"]],
           on="customer_id", how="inner")
)

print("Строк после соединения:", data.shape[0])
data.head()

Строк после соединения: 14572


,customer_id,first_name,last_name,address_id,city_id,city,country_id,country,payment_id,amount
0,524,Jared,Ely,530,419,Purwakarta,45,Indonesia,18202,1.99
1,524,Jared,Ely,530,419,Purwakarta,45,Indonesia,18203,4.99
2,524,Jared,Ely,530,419,Purwakarta,45,Indonesia,18204,2.99
3,524,Jared,Ely,530,419,Purwakarta,45,Indonesia,21950,2.99
4,524,Jared,Ely,530,419,Purwakarta,45,Indonesia,21951,4.99


Число строк совпало с количеством положительных платежей — 14572. Это хорошая
проверка: раз соединения не изменили количество строк, значит ни одно из них
не размножило данные. Так и должно быть, потому что все они идут
«многие к одному» по ключам справочников.

Отдельно соберём имя клиента — в SQL это была конкатенация `first_name || ' ' || last_name`.

In [6]:
data["customer_name"] = data["first_name"] + " " + data["last_name"]
data[["customer_id", "customer_name", "city", "country", "amount"]].head()

,customer_id,customer_name,city,country,amount
0,524,Jared Ely,Purwakarta,Indonesia,1.99
1,524,Jared Ely,Purwakarta,Indonesia,4.99
2,524,Jared Ely,Purwakarta,Indonesia,2.99
3,524,Jared Ely,Purwakarta,Indonesia,2.99
4,524,Jared Ely,Purwakarta,Indonesia,4.99


## Шаг 4. GROUP BY — считаем показатели по клиенту

Аналог `GROUP BY` с двумя агрегатами: `COUNT(payment_id)` и `SUM(amount)`.
Используется именованная агрегация — она сразу задаёт названия колонок результата.

In [7]:
revenue = (
    data
    .groupby(["country", "city", "customer_id", "customer_name"], as_index=False)
    .agg(
        payments_count=("payment_id", "count"),
        total_amount=("amount", "sum"),
    )
)

print("Клиентов в результате:", revenue.shape[0])
revenue.head()

Клиентов в результате: 599


,country,city,customer_id,customer_name,payments_count,total_amount
0,Afghanistan,Kabul,218,Vera Mccoy,18,67.82
1,Algeria,Batna,441,Mario Cheatham,27,107.73
2,Algeria,Bchar,69,Judy Gray,23,89.77
3,Algeria,Skikda,176,June Carroll,32,151.68
4,American Samoa,Tafuna,320,Anthony Schwab,15,47.85


Здесь нужна одна важная поправка. В базе поле `amount` имеет тип `numeric` —
это точное десятичное число. В Pandas оно читается как число с плавающей запятой,
а такие числа хранятся в двоичном виде, и при суммировании накапливается
погрешность в последних знаках.

Это хорошо видно, если вывести суммы как есть:

In [8]:
# Суммы до округления — видны «хвосты» от двоичного представления
print(revenue["total_amount"].head(8).to_list())

[67.82000000000001, 107.73, 89.77000000000001, 151.68, 47.85, 93.80000000000001, 93.75, 99.68]


In [9]:
# Округляем до двух знаков — столько же хранится в базе
revenue["total_amount"] = revenue["total_amount"].round(2)
print(revenue["total_amount"].head(8).to_list())

[67.82, 107.73, 89.77, 151.68, 47.85, 93.8, 93.75, 99.68]


## Шаг 5. Оконная функция — ранг клиента внутри своей страны

`RANK() OVER (PARTITION BY country ORDER BY total_amount DESC)` в Pandas
собирается из двух частей: `groupby("country")` задаёт разбиение, а метод `rank`
считает ранг внутри каждой группы.

Параметр `method="min"` здесь принципиален. Именно он воспроизводит поведение
SQL-функции `RANK`: при совпадении сумм клиенты получают одинаковый ранг,
а следующий за ними ранг пропускается. По умолчанию Pandas усредняет ранги
при совпадениях, и результат разошёлся бы с SQL.

In [10]:
revenue["rank_in_country"] = (
    revenue
    .groupby("country")["total_amount"]
    .rank(method="min", ascending=False)
    .astype(int)
)

revenue.sort_values(["country", "rank_in_country"]).head()

,country,city,customer_id,customer_name,payments_count,total_amount,rank_in_country
0,Afghanistan,Kabul,218,Vera Mccoy,18,67.82,1
3,Algeria,Skikda,176,June Carroll,32,151.68,1
1,Algeria,Batna,441,Mario Cheatham,27,107.73,2
2,Algeria,Bchar,69,Judy Gray,23,89.77,3
4,American Samoa,Tafuna,320,Anthony Schwab,15,47.85,1


## Шаг 6. CASE — сегмент клиента

Конструкции `CASE WHEN ... THEN ... ELSE ... END` соответствует `np.where`.
Поскольку условий три, вызовы вкладываются друг в друга: внешний проверяет
первое условие, внутренний разбирает оставшиеся варианты.

In [11]:
revenue["segment"] = np.where(
    revenue["total_amount"] >= 150, "ключевой",
    np.where(revenue["total_amount"] >= 100, "постоянный", "обычный")
)

revenue["segment"].value_counts()

segment
обычный       303
постоянный    275
ключевой       21
Name: count, dtype: int64

## Шаг 7. ORDER BY и итоговый результат

In [12]:
result = (
    revenue[["country", "city", "customer_name", "payments_count",
             "total_amount", "rank_in_country", "segment"]]
    .sort_values(["country", "rank_in_country", "customer_name"])
    .reset_index(drop=True)
)

print("Итого строк:", result.shape[0])
result.head(12)

Итого строк: 599


,country,city,customer_name,payments_count,total_amount,rank_in_country,segment
0,Afghanistan,Kabul,Vera Mccoy,18,67.82,1,обычный
1,Algeria,Skikda,June Carroll,32,151.68,1,ключевой
2,Algeria,Batna,Mario Cheatham,27,107.73,2,постоянный
3,Algeria,Bchar,Judy Gray,23,89.77,3,обычный
4,American Samoa,Tafuna,Anthony Schwab,15,47.85,1,обычный
5,Angola,Benguela,Claude Herzog,20,93.80,1,обычный
6,Angola,Namibe,Martin Bales,25,93.75,2,обычный
7,Anguilla,South Hill,Bobby Boudreau,32,99.68,1,обычный
8,Argentina,Avellaneda,Jordan Archuleta,28,129.71,1,постоянный
9,Argentina,La Plata,Julia Flores,29,124.71,2,постоянный


In [13]:
# Как выглядит один конкретный рынок
result.query("country == 'India'").head(10)

,country,city,customer_name,payments_count,total_amount,rank_in_country,segment
178,India,Valparai,Mike Way,33,162.67,1,ключевой
179,India,Halisahar,Lena Jensen,30,154.70,2,ключевой
180,India,Bijapur,Tim Cary,34,154.66,3,ключевой
181,India,Bhilwara,Tonya Chapman,29,147.71,4,постоянный
182,India,Rae Bareli,Lori Wood,31,141.69,5,постоянный
183,India,Bhopal,Helen Harris,31,134.68,6,постоянный
184,India,Siliguri (Shiliguri),Brett Cornwell,30,130.70,7,постоянный
185,India,Purnea (Purnia),Bradley Motley,26,125.74,8,постоянный
186,India,Parbhani,John Farnsworth,26,123.74,9,постоянный
187,India,Vijayawada,Milton Howland,23,121.77,10,постоянный


## Шаг 8. Сверка с результатом SQL-запроса

Расчёт повторён средствами Pandas, но это ещё не значит, что он совпадает
с оригиналом. Проверим напрямую: результат SQL-запроса из урока 12 выгружен
в файл `data/sql_reference.csv`, сравним обе таблицы построчно.

In [14]:
sql_result = pd.read_csv("data/sql_reference.csv")

print("Строк в SQL-результате   :", sql_result.shape[0])
print("Строк в расчёте Pandas   :", result.shape[0])
print("Колонки совпадают        :", list(sql_result.columns) == list(result.columns))

Строк в SQL-результате   : 599
Строк в расчёте Pandas   : 599
Колонки совпадают        : True


In [15]:
# приводим обе таблицы к одному порядку строк и одному типу чисел
cols = list(result.columns)

from_sql = (sql_result[cols]
            .sort_values(["country", "rank_in_country", "customer_name"])
            .reset_index(drop=True))
from_pandas = (result[cols]
               .sort_values(["country", "rank_in_country", "customer_name"])
               .reset_index(drop=True))

from_sql["total_amount"] = from_sql["total_amount"].round(2)
from_pandas["total_amount"] = from_pandas["total_amount"].round(2)

identical = from_sql.equals(from_pandas)
print("Таблицы полностью совпадают:", identical)

Таблицы полностью совпадают: True


In [16]:
# если где-то есть расхождения — покажем их поимённо
diff_report = {}
for c in cols:
    mismatches = (from_sql[c] != from_pandas[c]).sum()
    diff_report[c] = mismatches

pd.Series(diff_report, name="расхождений")

country            0
city               0
customer_name      0
payments_count     0
total_amount       0
rank_in_country    0
segment            0
Name: расхождений, dtype: int64

## Выводы

Расчёт, изначально написанный на SQL, полностью воспроизведён средствами Pandas.
Построчная сверка с выгруженным результатом SQL-запроса показала совпадение
по всем 599 строкам и всем семи колонкам.

Соответствие конструкций оказалось почти дословным: `INNER JOIN` превращается
в `merge`, `WHERE` в `query`, `GROUP BY` с агрегатами в `groupby().agg()`,
`CASE` в `np.where`, `ORDER BY` в `sort_values`.

Два места потребовали внимания, и оба не видны из самой формулировки задачи.

Первое — тип данных. В базе суммы хранятся как точные десятичные числа,
а Pandas читает их как числа с плавающей запятой. Без округления до двух знаков
суммы разошлись бы с SQL в последних знаках, и сравнение не прошло бы,
хотя расчёт был бы по сути верным.

Второе — параметр `method="min"` у `rank`. По умолчанию Pandas при совпадении
значений усредняет ранги и выдаёт дробные номера, тогда как SQL-функция `RANK`
присваивает одинаковый ранг и пропускает следующий. Эти две логики расходятся
ровно там, где в данных есть клиенты с одинаковой суммой, — и именно такие
случаи было бы легче всего не заметить.